# Trend-following with smaller positions after overextended moves

This code lays the foundation for a trend-following strategy that holds smaller positions after an unusually large move in the trend's direction.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

NOTEBOOK_FILE = globals().get("__vsc_ipynb_file__")
WORKSPACE_FOLDER = Path(NOTEBOOK_FILE).resolve().parents[1] if NOTEBOOK_FILE else Path.cwd()
sys.path.insert(0, str(WORKSPACE_FOLDER / "src"))

from backtest.evaluation import calc_key_figures, plot_key_figures

DATE_COL = "AsOfDate"
PRICES = pl.read_csv(WORKSPACE_FOLDER / "prices" / "example_prices.csv", try_parse_dates=True).rename({"dates": DATE_COL})
INSTRUMENTS = [c for c in PRICES.columns if c != DATE_COL]

### Strategy summarized
* Estimate the volatility of the assets at time $T$ from the last $N$ days:
    $$\sigma_{T,j} = \sqrt{\frac{1}{N}\sum_{k=T-N+1}^{T} r_{k,j}^2}$$

* Calculate a fast and a slow exponentially weighted moving average (EWMA) of the price, with spans $M$ and $4M$. For a span $s$:

    $$\text{EWMA}^{(s)}_{T,j} = \alpha_s P_{T,j} + (1 - \alpha_s)\, \text{EWMA}^{(s)}_{T-1,j}, \qquad \alpha_s = \frac{2}{s + 1}$$

* Calculate the trend signal from the difference between the two averages:

    $$S^{\text{trend}}_{T, j} = \text{sign}\left(\text{EWMA}^{(M)}_{T,j} - \text{EWMA}^{(4M)}_{T,j}\right)$$

* Measure how unusual the move over the last $L$ days is:

    $$z_{T,j} = \frac{P_{T,j} - P_{T-L,j}}{\sigma_{T-L,j}\sqrt{L}}$$

    We use the volatility from before the move, $\sigma_{T-L,j}$, so that the move does not inflate its own yardstick. The factor $\sqrt{L}$ is there because the variance of a sum of independent daily returns grows with the number of days.

* Calculate the reversal signal, which bets against moves larger than a threshold $\theta$:

    $$S^{\text{rev}}_{T, j} = \begin{cases} -\text{sign}(z_{T,j}) & \text{if } |z_{T,j}| > \theta \\ 0 & \text{otherwise} \end{cases}$$

* Calculate the signal, which follows the trend but holds a smaller position after an unusually large move in the trend's direction (an overextended move). This happens when the reversal signal opposes the trend signal, and the share of the position removed is $\lambda$:

    $$X_{T, j} = \begin{cases} (1 - \lambda)\, S^{\text{trend}}_{T,j} & \text{if } S^{\text{rev}}_{T,j} = -S^{\text{trend}}_{T,j} \\ S^{\text{trend}}_{T,j} & \text{otherwise} \end{cases}$$

    Note that with $\lambda = 0.5$, the signal for a given asset will be $\pm 1$, or $\pm 0.5$ when the move is overextended (apart from exact ties).

* Calculate the positions $\tilde X_{T,j}$ that would make each asset contribute risk in proportion to $|X_{T,j}|$ (ignoring correlations between assets):
    $$\tilde X_{T,j} = \frac{X_{T,j}}{\sigma_{T,j}}$$

* The returns from  these positions need to be calculated taking into account the two-day lag:
    $$R_T = \sum_j \tilde X_{T-2,j}\, r_{T,j}$$

* The next step is to calculate the portfolio risk from holding these positions:
    $$\hat\sigma_T = \text{std}\left(R_{T-N+1}, \dots, R_T\right)$$

* We can then calculate the final portfolio positions as:
    $$\hat X_{T,j} = \frac{\tilde X_{T,j}}{\hat\sigma_T}$$

    Dividing by $\sigma_{T,j}$ balances risk between assets, and dividing by $\hat\sigma_T$ keeps the risk of the whole strategy steady over time. Defining the risk of an asset as the size of our position multiplied by its volatility:

    $$\text{risk}_{T+2,j} = \sigma^{\text{true}}_{T+2,j} \cdot |\hat X_{T,j}| = \sigma^{\text{true}}_{T+2,j} \cdot \frac{|X_{T,j}|}{\sigma_{T,j}\,\hat\sigma_T} \approx \frac{|X_{T,j}|}{\hat\sigma_T}$$

    Every asset therefore carries risk in proportion to the size of its signal, $|X_{T,j}|/\hat\sigma_T$.

* The realized return from the portfolio will be:

    $$\hat R_{T+2} = \sum_j \hat X_{T,j}\, r_{T+2,j} = \frac{1}{\hat\sigma_T}\sum_j \tilde X_{T,j}\, r_{T+2,j} = \frac{R_{T+2}}{\hat\sigma_T}$$

    Let $\sigma^{\text{true}}_{T+2}$ be the true, unobserved volatility of $R_{T+2}$, given the data through day $T$. Since $\hat\sigma_T$ is known on day $T$, it acts as a constant:

    $$\text{Var}\left(\hat R_{T+2} \mid \text{data through } T\right) = \frac{\text{Var}\left(R_{T+2} \mid \text{data through } T\right)}{\hat\sigma_T^2} = \frac{\left(\sigma^{\text{true}}_{T+2}\right)^2}{\hat\sigma_T^2} \approx 1$$

    The final return therefore has a standard deviation of about 1, provided $\hat\sigma_T$ is a good forecast of $\sigma^{\text{true}}_{T+2}$.

* The combination comes from staying with the trend while taking less risk when a move may have gone too far. The return on asset $j$ is:

    $$\hat X_{T,j}\, r_{T+2,j} = \frac{X_{T,j}\, r_{T+2,j}}{\sigma_{T,j}\,\hat\sigma_T}$$

    Shrinking the position only helps if the trend earns less after an overextended move than at other times:

    $$E\left[S^{\text{trend}}_{T,j}\, r_{T+2,j} \,\middle|\, S^{\text{rev}}_{T,j} = -S^{\text{trend}}_{T,j}\right] < E\left[S^{\text{trend}}_{T,j}\, r_{T+2,j} \,\middle|\, S^{\text{rev}}_{T,j} = 0\right]$$

    This requires overextended moves to lose momentum or partly reverse. The example data leans this way but does not establish it. With $L = 5$ and $\theta = 2$, the return in the trend's direction averages about $0.00$ daily standard deviations after an overextended move and about $0.02$ when no large move has occurred, and the difference is not statistically significant ($t \approx -0.6$).

* Trading incurs a transaction cost proportional to the size of each order. As calculated in `evaluation.py`, the net return of the strategy is:

    $$\hat R^{\text{net}}_T = \sum_j \left[\hat X_{T-2,j}\, r_{T,j} - c\, s_{T,j} \left|\hat X_{T-2,j} - \hat X_{T-3,j}\right|\right]$$

    Here $c = 0.02$, and $s_{T,j}$ is the rolling 100-day standard deviation of the daily returns of asset $j$, using data through day $T-1$. This net return is what the Adjusted Sharpe Ratio is evaluated on.

    Costs arise in three ways. The largest orders come from trend reversals, when $S^{\text{trend}}_{T,j}$ flips sign. Smaller orders occur when a move becomes or stops being overextended, changing the position by $\lambda$, and almost every day as $\sigma_{T,j}$ and $\hat\sigma_T$ change. A larger $\lambda$ or a lower $\theta$ resizes positions more often, which trades more.


In [ ]:
def calc_positions(M_trend, M_rev, theta, shrink, prices, N=100):
    """
    Calculate the final positions of the trend strategy that shrinks positions after overextended moves.

    Parameters:
    M_trend (int): The span of the fast EWMA, in days. The slow EWMA has a span of 4 * M_trend.
    M_rev (int): The length of the move used for the reversal signal, in days.
    theta (float): How large a move counts as unusual, in standard deviations.
    shrink (float): The share of the position to remove when a move is overextended.
    prices (pl.DataFrame): DataFrame containing a date column and one price column per instrument.
    N (int): The window for the volatility and portfolio risk estimates, in days.

    Returns:
    pl.DataFrame: The date column and the final position of each instrument. Positions are null
    until there is enough data for all windows.
    """
    instruments = [c for c in prices.columns if c != DATE_COL]
    returns = prices.select(pl.col(instruments).diff())

    # Asset volatility: sigma_T = sqrt(mean of r^2 over the last N days)
    sigma = returns.select(pl.col(instruments).pow(2).rolling_mean(N).sqrt())

    # Fast and slow EWMAs of the price, with spans M_trend and 4 * M_trend
    ewma_fast = prices.select(pl.col(instruments).ewm_mean(span=M_trend, adjust=False, min_samples=M_trend))
    ewma_slow = prices.select(pl.col(instruments).ewm_mean(span=4 * M_trend, adjust=False, min_samples=4 * M_trend))

    # Trend signal: sign(EWMA_fast_T - EWMA_slow_T)
    trend = (ewma_fast - ewma_slow).select(pl.all().sign())

    # Size of the move: z_T = (P_T - P_{T-M_rev}) / (sigma_{T-M_rev} * sqrt(M_rev))
    z = prices.select(pl.col(instruments).diff(M_rev)) / sigma.select(pl.all().shift(M_rev) * M_rev**0.5)

    # Reversal signal: -sign(z_T) if |z_T| > theta, otherwise 0
    reversal = z.select(-pl.all().sign() * (pl.all().abs() > theta).cast(pl.Float64))

    # A move is overextended if it is unusually large in the trend's direction, so the reversal signal opposes the trend
    size = (reversal * trend).select((pl.all() < 0).cast(pl.Float64) * (-shrink) + 1)

    # Signal: X_T = trend_T * (1 - shrink) if overextended, otherwise trend_T
    X = trend * size

    # Equal-risk positions: X_tilde_T = X_T / sigma_T
    X_tilde = X / sigma

    # Portfolio return with the two-day lag: R_T = sum over j of X_tilde_{T-2} * r_T
    asset_returns = X_tilde.select(pl.all().shift(2)) * returns
    R = asset_returns.select(
        pl.when(pl.all_horizontal(pl.all().is_null()))
        .then(None)
        .otherwise(pl.sum_horizontal(pl.all()))
        .alias("R")
    )["R"]

    # Portfolio risk: sigma_hat_T = std of R over the last N days
    sigma_hat = R.rolling_std(N)

    # Final positions: X_hat_T = X_tilde_T / sigma_hat_T
    X_hat = X_tilde.select(pl.all() / sigma_hat).fill_nan(None)

    return X_hat.with_columns(prices[DATE_COL]).select(DATE_COL, *instruments)


positions = calc_positions(16, 5, 2.0, 0.5, PRICES)

### Evaluation
We evaluate the positions with the tools in `evaluation.py`, including its approximation of transaction costs (slippage).

In [ ]:
plot_key_figures(positions, PRICES)

#### Key figures
The default key figures as a dict. See the documentation in `evaluation.py` for what each one means.

In [ ]:
calc_key_figures(positions, PRICES)

#### Cost sensitivity
How the Sharpe ratio changes with the cost coefficient. The default in `evaluation.py` is `costs=-0.02`.

In [ ]:
cost_coeffs = np.linspace(0, -0.1, 10)
sharpe_per_cost_coef = [
    calc_key_figures(positions, PRICES, costs=float(cost), key_figures=["sharpe"])["sharpe"]
    for cost in cost_coeffs
]
fig, ax = plt.subplots()
ax.plot(cost_coeffs, sharpe_per_cost_coef)
ax.set_title("Cost sensitivity")
ax.set_ylabel("Sharpe after costs")
ax.set_xlabel("Cost coefficient")
ax.grid()

### Leakage test
Positions on day $T$ may only use data up to and including day $T$. We test this in the two ways described in the instructions: by using a shorter history, and by changing the prices after day $T$. In both cases, the positions up to day $T$ must stay identical.

In [ ]:
def check_no_leakage(position_func, prices, days_removed=20):
    """
    Raise an AssertionError if the positions from position_func depend on future data.

    Parameters:
    position_func (callable): Takes a prices DataFrame and returns a positions DataFrame.
    prices (pl.DataFrame): DataFrame containing a date column and one price column per instrument.
    days_removed (int): The number of days to remove in the shorter-history test.
    """
    instruments = [c for c in prices.columns if c != DATE_COL]
    full = position_func(prices)

    # Test 1: positions from a shorter history must equal those from the full history
    truncated = position_func(prices.head(prices.height - days_removed))
    assert truncated.equals(full.head(truncated.height)), "Leakage: shorter history changes earlier positions"

    # Test 2: changing prices after day T must not change positions up to and including day T
    T = prices.height // 2
    is_future = pl.int_range(pl.len()) > T
    altered = prices.with_columns(
        pl.when(is_future).then(pl.col(c) * 1.5 + 100).otherwise(pl.col(c)).alias(c) for c in instruments
    )
    assert position_func(altered).head(T + 1).equals(full.head(T + 1)), "Leakage: future prices change earlier positions"


check_no_leakage(lambda prices: calc_positions(16, 5, 2.0, 0.5, prices), PRICES)
print("Leakage tests passed")